# Track C — Deep learning

This notebook demonstrates the requested FFNN, TextCNN, and DistilBERT implementations. All models use the shared per-repository evaluation contract. The command-line runner in `examples/deep_learning.py` is the reproducible way to generate final artifacts.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_ROOT

## 1. Load one repository

The competition trains a separate classifier for each project. Raw text, title weight 3, and 400-word truncation are the training-only preprocessing choice used for the FFNN and CNN.

In [ ]:
from ai4se.service import IssueDataService

service = IssueDataService.from_loader(kind='memory')
react = service.prepare(
    'train', repo='facebook/react', level='raw', max_words=400, title_weight=3
)
texts = [issue.text for issue in react]
labels = [issue.label for issue in react]
len(texts), sorted(set(labels))

## 2. FFNN and TextCNN

The internal validation split is created before TF-IDF or vocabulary fitting and keeps duplicate text groups together. Reduce `epochs` for a quick smoke test.

In [ ]:
from dataclasses import replace

from ai4se.deep_learning import (
    CNNConfig,
    CNNTextClassifier,
    FFNNConfig,
    FeedForwardTextClassifier,
    TrainingConfig,
    plot_learning_history,
)

training = TrainingConfig(epochs=20, patience=5, device='auto')
ffnn = FeedForwardTextClassifier(FFNNConfig(training=training)).fit(texts, labels)
plot_learning_history(ffnn.history_.to_dict(), title='FFNN — facebook/react');

In [ ]:
cnn = CNNTextClassifier(CNNConfig(training=training)).fit(texts, labels)
plot_learning_history(cnn.history_.to_dict(), title='TextCNN — facebook/react');

## 3. DistilBERT fine-tuning

DistilBERT is selected instead of RoBERTa to reduce memory and runtime on Colab. Enable the next cell only with the `dl` dependencies installed; a GPU is strongly recommended.

In [ ]:
RUN_TRANSFORMER = False

if RUN_TRANSFORMER:
    from ai4se.deep_learning import DistilBERTConfig, DistilBERTTextClassifier

    transformer_issues = service.prepare(
        'train', repo='facebook/react', level='light', title_weight=1
    )
    transformer_training = TrainingConfig(
        epochs=4, batch_size=8, learning_rate=2e-5, patience=2, device='auto'
    )
    transformer = DistilBERTTextClassifier(
        DistilBERTConfig(training=transformer_training)
    ).fit(
        [issue.text for issue in transformer_issues],
        [issue.label for issue in transformer_issues],
    )
    plot_learning_history(
        transformer.history_.to_dict(), title='DistilBERT — facebook/react'
    );

## 4. Generate final results

Run these commands from the repository root. Cross-validation must be used for tuning; official mode is the final held-out evaluation.

```bash
make deep-neural
make deep-transformer
python examples/deep_learning.py --mode cross-validation --models ffnn cnn --plots
```